# Fake News Detection — Model Training
Trains DistilBERT / RoBERTa / XLNet on `Dataset_Clean.csv`.

**Runtime:** GPU (T4 recommended). Go to `Runtime → Change runtime type → T4 GPU`.

In [ ]:
# ── 1. Mount Google Drive (upload Dataset_Clean.csv there first) ──────────
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ── 2. Install dependencies ───────────────────────────────────────────────
!pip install -q transformers datasets scikit-learn accelerate wandb

In [ ]:
# ── 3. Clone repo (or upload files manually) ──────────────────────────────
# Option A: clone from GitHub
# !git clone https://github.com/YOUR_USERNAME/fake-news-detection.git
# %cd fake-news-detection

# Option B: already in /content — set paths manually
import os
PROJECT_ROOT = '/content/fake-news-detection'  # adjust if needed
DATA_CSV = f'{PROJECT_ROOT}/data/processed/Dataset_Clean.csv'
MODELS_DIR = f'{PROJECT_ROOT}/models'
os.makedirs(MODELS_DIR, exist_ok=True)

In [ ]:
# ── 4. Config ─────────────────────────────────────────────────────────────
MODEL_KEY   = 'distilbert'   # 'distilbert' | 'roberta' | 'xlnet'
EPOCHS      = 3
BATCH_SIZE  = 32             # increase if GPU has headroom
LR          = 2e-5
MAX_LENGTH  = 256
USE_WANDB   = True
WANDB_API_KEY = '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'

MODEL_NAMES = {
    'distilbert': 'distilbert-base-uncased',
    'roberta':    'roberta-base',
    'xlnet':      'xlnet-base-cased',
}
MODEL_NAME = MODEL_NAMES[MODEL_KEY]
OUTPUT_DIR = f'{MODELS_DIR}/{MODEL_KEY}'
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'Training {MODEL_KEY} ({MODEL_NAME})')

In [ ]:
# ── 5. Load & split dataset ───────────────────────────────────────────────
import pandas as pd
import re
from sklearn.model_selection import train_test_split
from datasets import Dataset, DatasetDict

LABEL2ID = {'True': 0, 'Fake': 1, 'Satire': 2, 'Bias': 3}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}

def clean_text(text):
    text = str(text).lower().strip()
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'[^\w\s.,!?]', ' ', text)
    return re.sub(r'\s+', ' ', text).strip()

df = pd.read_csv(DATA_CSV, low_memory=False)
df['label_text'] = df['label_text'].astype(str).str.strip().str.capitalize()
df = df[df['label_text'].isin(LABEL2ID)].copy()
df['content'] = df['content'].fillna('').astype(str)
df['title']   = df['title'].fillna('').astype(str)
df['text']    = df.apply(lambda r: r['content'] if len(r['content']) > 30 else r['title'], axis=1)
df['text']    = df['text'].apply(clean_text)
df = df[df['text'].str.len() > 10].copy()
df['label']   = df['label_text'].map(LABEL2ID).astype(int)
df = df[['text', 'label']].reset_index(drop=True)

print(f'Total rows: {len(df):,}')
print(df['label'].value_counts().rename(ID2LABEL))

train_df, temp_df = train_test_split(df, test_size=0.20, stratify=df['label'], random_state=42)
val_df,  test_df  = train_test_split(temp_df, test_size=0.50, stratify=temp_df['label'], random_state=42)
print(f'Train: {len(train_df):,}  Val: {len(val_df):,}  Test: {len(test_df):,}')

raw = DatasetDict({
    'train':      Dataset.from_pandas(train_df, preserve_index=False),
    'validation': Dataset.from_pandas(val_df,   preserve_index=False),
    'test':       Dataset.from_pandas(test_df,  preserve_index=False),
})

In [ ]:
# ── 6. Tokenize ───────────────────────────────────────────────────────────
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(batch['text'], padding='max_length', truncation=True, max_length=MAX_LENGTH)

tokenized = raw.map(tokenize, batched=True, batch_size=512, remove_columns=['text'], desc='Tokenizing')
tokenized.set_format('torch')
print(tokenized)

In [ ]:
# ── 7. Model ──────────────────────────────────────────────────────────────
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=4,
    id2label=ID2LABEL,
    label2id=LABEL2ID,
    ignore_mismatched_sizes=True,
)
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')

In [ ]:
# ── 8. Metrics ────────────────────────────────────────────────────────────
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        'accuracy':    round(accuracy_score(labels, preds), 4),
        'f1_macro':    round(f1_score(labels, preds, average='macro',    zero_division=0), 4),
        'f1_weighted': round(f1_score(labels, preds, average='weighted', zero_division=0), 4),
    }

In [ ]:
# ── 9. Train ──────────────────────────────────────────────────────────────
import torch
from transformers import TrainingArguments, Trainer, EarlyStoppingCallback

if USE_WANDB:
    import wandb
    wandb.login(key=WANDB_API_KEY)
    wandb.init(project='fake-news-detection', name=f'{MODEL_KEY}-colab')

training_args = TrainingArguments(
    output_dir=f'{OUTPUT_DIR}/checkpoints',
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE * 2,
    learning_rate=LR,
    weight_decay=0.01,
    warmup_ratio=0.06,
    evaluation_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1_macro',
    greater_is_better=True,
    save_total_limit=2,
    logging_steps=100,
    fp16=torch.cuda.is_available(),
    report_to='wandb' if USE_WANDB else 'none',
    run_name=f'{MODEL_KEY}-colab',
    push_to_hub=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized['train'],
    eval_dataset=tokenized['validation'],
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

trainer.train()

In [ ]:
# ── 10. Evaluate on test set ──────────────────────────────────────────────
from sklearn.metrics import classification_report, confusion_matrix

preds_out = trainer.predict(tokenized['test'])
preds  = np.argmax(preds_out.predictions, axis=-1)
labels = preds_out.label_ids

print(classification_report(labels, preds, target_names=list(LABEL2ID.keys()), zero_division=0))
print('Confusion Matrix:')
print(confusion_matrix(labels, preds))

In [ ]:
# ── 11. Save model ────────────────────────────────────────────────────────
import json

trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f'Model saved to {OUTPUT_DIR}')

# Save metrics
from sklearn.metrics import classification_report as cr
report = cr(labels, preds, target_names=list(LABEL2ID.keys()), output_dict=True, zero_division=0)
with open(f'{OUTPUT_DIR}/metrics.json', 'w') as f:
    json.dump(report, f, indent=2)
print('metrics.json saved')

In [ ]:
# ── 12. Download model to local machine ───────────────────────────────────
# Zip the model folder and download it
import shutil
zip_path = f'/content/{MODEL_KEY}_model'
shutil.make_archive(zip_path, 'zip', OUTPUT_DIR)

from google.colab import files
files.download(f'{zip_path}.zip')
print(f'Downloaded {MODEL_KEY}_model.zip — extract to models/{MODEL_KEY}/')